In [ ]:
!pip install -r ../scripts/requirements.txt


# 00 — Readiness check

Quick preflight for the rest of this quickstart. Before running `01_agent.ipynb`, `02_router_cache.ipynb`, or `03_async_work_queue.ipynb`, this notebook verifies the two external dependencies those notebooks assume are live:

1. **Redis** — reachable at `REDIS_URL` and able to `PING`.
2. **MaaS model endpoints** — the *simple* and *complex* chat-completion endpoints respond to a minimal `chat.completions` call with valid credentials.

If every check is ✅ the notebooks downstream will work end-to-end.

## 2. Load environment

Values come from `../../.env` (see the repo README). The `SIMPLE_*` and `COMPLEX_*` triplets point at two distinct MaaS models so we can route cheap questions cheaply and hard questions to the reasoning model.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parents[1]
load_dotenv(REPO_ROOT / ".env")

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379")

SIMPLE_ENDPOINT = os.environ.get("SIMPLE_MODEL_ENDPOINT")
SIMPLE_KEY = os.environ.get("SIMPLE_MODEL_KEY")
SIMPLE_MODEL = os.environ.get("SIMPLE_MODEL_NAME", "qwen3-14b")

COMPLEX_ENDPOINT = os.environ.get("COMPLEX_MODEL_ENDPOINT")
COMPLEX_KEY = os.environ.get("COMPLEX_MODEL_KEY")
COMPLEX_MODEL = os.environ.get("COMPLEX_MODEL_NAME", "deepseek-r1-distill-qwen-14b")

def _base_url(endpoint: str | None) -> str | None:
    """Normalize a chat-completions endpoint to the /v1 base URL."""
    if not endpoint:
        return None
    trimmed = endpoint.rstrip('/')
    for suffix in ('/chat/completions', '/completions'):
        if trimmed.endswith(suffix):
            return trimmed[: -len(suffix)]
    return trimmed

SIMPLE_BASE_URL = _base_url(SIMPLE_ENDPOINT)
COMPLEX_BASE_URL = _base_url(COMPLEX_ENDPOINT)

def _mask(val: str | None) -> str:
    if not val:
        return "(unset)"
    return val[:4] + "…" + val[-4:] if len(val) > 10 else "***"

print(f"redis   : {REDIS_URL}")
print(f"simple  : {SIMPLE_MODEL}  @  {SIMPLE_BASE_URL}  key={_mask(SIMPLE_KEY)}")
print(f"complex : {COMPLEX_MODEL}  @  {COMPLEX_BASE_URL}  key={_mask(COMPLEX_KEY)}")

RESULTS = {}  # populated by the checks below


redis   : redis://localhost:6379
simple  : qwen3-14b  @  https://litellm-prod.apps.maas.redhatworkshops.io/v1  key=sk-y…9j2w
complex : deepseek-r1-distill-qwen-14b  @  https://litellm-prod.apps.maas.redhatworkshops.io/v1  key=sk-H…OgSg


## 3. Redis

A one-shot `PING` proves the URL parses, TLS/auth are correct, and the server answers. This is the same client the router, semantic cache, LangGraph checkpointer, and work queue all share downstream.

In [2]:
import redis

try:
    r = redis.Redis.from_url(REDIS_URL, socket_connect_timeout=5)
    pong = r.ping()
    info = r.info(section="server")
    version = info.get("redis_version") or info.get("valkey_version", "?")
    RESULTS["redis"] = (True, f"PING={pong}  version={version}")
    print(f"✅ redis reachable — version {version}")
except Exception as exc:
    RESULTS["redis"] = (False, f"{type(exc).__name__}: {exc}")
    print(f"❌ redis unreachable — {exc}")


✅ redis reachable — version 8.6.2


## 4. MaaS endpoints

One minimal chat completion per model (1 output token) confirms the base URL, API key, and model name all resolve on the gateway. Failure here is almost always a typo in the endpoint or an expired key.

In [3]:
from openai import OpenAI

def check_model(label: str, base_url: str | None, api_key: str | None, model: str) -> tuple[bool, str]:
    if not base_url or not api_key:
        return False, "endpoint or key missing from .env"
    try:
        client = OpenAI(base_url=base_url, api_key=api_key, timeout=15.0)
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=1,
        )
        return True, f"served_by={resp.model}  id={resp.id}"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"

for label, base, key, model in [
    ("simple", SIMPLE_BASE_URL, SIMPLE_KEY, SIMPLE_MODEL),
    ("complex", COMPLEX_BASE_URL, COMPLEX_KEY, COMPLEX_MODEL),
]:
    ok, detail = check_model(label, base, key, model)
    RESULTS[f"maas:{label}"] = (ok, detail)
    print(f"{'✅' if ok else '❌'} {label:<7} {model}  —  {detail}")


✅ simple  qwen3-14b  —  served_by=qwen3-14b  id=chatcmpl-bb40e3ab9cd84811bc047b158217b3fc


✅ complex deepseek-r1-distill-qwen-14b  —  served_by=deepseek-r1-distill-qwen-14b  id=chatcmpl-8fc018a3585042c89930855708058f9f


## 5. Summary

If any row is ❌, fix it before moving on — every downstream notebook assumes the checks above pass.

In [4]:
print(f"{'service':<16} {'status':<8} detail")
print("-" * 72)
all_ok = True
for name, (ok, detail) in RESULTS.items():
    status = "✅ ok" if ok else "❌ fail"
    if not ok:
        all_ok = False
    print(f"{name:<16} {status:<8} {detail}")
print()
print("READY — run 01/02/03." if all_ok else "NOT READY — fix the ❌ rows first.")
assert all_ok, "readiness check failed"


service          status   detail
------------------------------------------------------------------------
redis            ✅ ok     PING=True  version=8.6.2
maas:simple      ✅ ok     served_by=qwen3-14b  id=chatcmpl-bb40e3ab9cd84811bc047b158217b3fc
maas:complex     ✅ ok     served_by=deepseek-r1-distill-qwen-14b  id=chatcmpl-8fc018a3585042c89930855708058f9f

READY — run 01/02/03.


In [ ]:
import os

# Check all ENV Variables needed in subsequent notebooks
for var in ["SIMPLE_API_KEY", "COMPLEX_API_KEY", "MODEL_ENDPOINT", "COMPLEX_MODEL_NAME", "SIMPLE_MODEL_NAME", "REDIS_URL"]:
    if not os.environ.get(var):
        print(f"⚠️ Environment variable {var} is not set. Please set it in the notebook or in the environment.")


In [ ]:
import redis
REDIS_URL = os.environ.get("REDIS_URL")

# Simple check if REDIS is running
try:
    r = redis.Redis.from_url(REDIS_URL)
    r.ping()
    print("✅ REDIS is running and reachable!")
except Exception as e:
    raise RuntimeError(f"❌ Cannot connect to REDIS at {REDIS_URL}: {e}")

In [ ]:
import os


def _chat_completions_url(endpoint: str | None) -> str | None:
    """Env may be OpenAI base (/v1) or full chat URL; requests.post needs .../chat/completions."""
    if not endpoint:
        return None
    trimmed = endpoint.rstrip("/")
    if trimmed.endswith("/chat/completions"):
        return trimmed
    return f"{trimmed}/chat/completions"


_complex_ep = os.environ.get("COMPLEX_MODEL_ENDPOINT") or os.environ.get("MODEL_ENDPOINT")
_simple_ep = os.environ.get("SIMPLE_MODEL_ENDPOINT") or os.environ.get("MODEL_ENDPOINT")

model_configs = [
    {
        "name": os.environ.get("COMPLEX_MODEL_NAME", "deepseek-r1-distill-qwen-14b"),
        "endpoint": _chat_completions_url(_complex_ep),
        "key": os.environ.get("COMPLEX_MODEL_KEY") or os.environ.get("MODEL_API_KEY"),
    },
    {
        "name": os.environ.get("SIMPLE_MODEL_NAME", "qwen3-14b"),
        "endpoint": _chat_completions_url(_simple_ep),
        "key": os.environ.get("SIMPLE_MODEL_KEY") or os.environ.get("MODEL_API_KEY"),
    },
]


def check_llm_availability(config):
    model_name = config.get("name")
    endpoint = config.get("endpoint")
    api_key = config.get("key")

    if not (model_name and endpoint and api_key):
        print(f"⚠️ Missing key information for model: name={model_name}, endpoint={endpoint}, key={'******' if api_key else None}")
        return False

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": "Hello, world!"}]
    }
    try:
        import requests
        resp = requests.post(endpoint, headers=headers, json=payload, timeout=10)
        try:
            resp_json = resp.json()
        except Exception as je:
            resp_json = {}
        if resp.status_code == 200 and isinstance(resp_json, dict) and "choices" in resp_json:
            print(f"✅ Model '{model_name}' is available.")
            return True
        else:
            print(f"❌ Model '{model_name}' unavailable or error: {resp.text}")
            return False
    except Exception as e:
        print(f"❌ Exception when testing model '{model_name}': {e}")
        return False

available_models = {}
for config in model_configs:
    model_name = config.get("name")
    if model_name:
        available_models[model_name] = check_llm_availability(config)

if available_models:
    print("\nModels available:")
    for model_name, is_available in available_models.items():
        print(f"  {model_name}: {'✅' if is_available else '❌'}")
else:
    print("No models available expecting to be available.")
